# Citation Network Analysis using Graph Neural Networks

## Install Instructions

We are going to use Google Colab for this tutorial, since we really need GPU access. To do this:
1. Go to https://colab.research.google.com/ (and make sure you are logged in with your Google account)
2. Click on "Upload", then choose "Github" and enter the URL of this repository: https://github.com/SteinhardtMU/ML2026
3. Scroll down to the "Week9" folder and open the "Citation_Network_GNN.ipynb" notebook
4. In the top toolbar, click on Connect
5. Select "Change runtime type"
6. Select "T4 GPU" from the menu, and click "Save"
7. Run the below line of code inside a notebook cell to get the code and data

In [ ]:
!rm -rf ML2026
!git clone https://github.com/SteinhardtMU/ML2026.git

Once you've done that, let's install the necessary libraries:

In [ ]:
# Colab GPU runtimes already have torch installed; just install the extras
!pip install ogb lightning torch_geometric requests scikit-learn
!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

## Intro

We are going to use graph neural networks to analyse a real citation network built around a researcher of your choice. We will look at two tasks:

**Task 1: Paper ranking.** Given the graph of citations and co-authorships, can a GNN learn a better measure of paper influence than raw citation count? Intuitively it should - a paper cited by highly-cited papers should rank higher than one with the same raw count but cited only by obscure works.

**Task 2: Link prediction.** Can the GNN predict which citations will appear in the *future*, having only seen the past?  The same idea is used to suggest new connections in social networks.

Along the way we will also look at whether both citation count and GNN scores depend on the year a paper was published.  Younger papers are obviously disadvantaged, and it will be interesting to see whether the GNN learns to correct for this.

The graph we will build has the following structure:

The **nodes** are papers, with features given by `graph.x`:
- `age`: how many years ago the paper was published
- `is_seed`: whether this is one of the author's own papers (vs. a cited neighbour)
- `log_in_degree`: log of how many papers in our graph cite this one
- `log_out_degree`: log of how many papers this one cites within our graph

Note that we deliberately do NOT include citation count as a node feature - if we did, the GNN could just copy it through without using the graph structure at all!

There are two types of **edges** (with `graph.edge_type`):
- Citation edges (type 0): paper A cites paper B
- Co-authorship edges (type 1): two of the author's papers share a co-author

The **label** for each node is `graph.y`: log(citation count + 1). The GNN has access to age and graph structure, so it can in principle learn to correct for age bias itself.

## Fetch the data

First, let's set up our imports and a caching system. The Semantic Scholar API is free but rate-limited, so we save everything to disk the first time and load from cache on subsequent runs. This means the first run will take 10-15 minutes, but after that it's instant.

In [ ]:
import torch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

# Required for PyTorch >= 2.6: allowlist PyG classes used in dataset files
torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])

import requests
import time
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.stats import spearmanr, kendalltau
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score

from torch import nn
import torch.nn.functional as F
import lightning as lit
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, aggr as pyg_aggr
from torch_geometric.utils import scatter, negative_sampling
scatter_add = lambda src, index, dim, dim_size: scatter(src, index, dim=dim, dim_size=dim_size, reduce='sum')

print(f"PyTorch:  {torch.__version__}")
print(f"PyG:      {torch_geometric.__version__}")
print(f"GPU:      {torch.cuda.is_available()}")

In [ ]:
# ── Settings ───────────────────────────────────────────────────────────────
AUTHOR_NAME  = "Nima Arkani-Hamed"   # Change this to your advisor!
TOP_N        = 50                     # Keep the top N papers by citation count
SPLIT_YEAR   = 2020                   # Past/future split for link prediction
CACHE_DIR    = "scholar_cache"        # Fetched data saved here
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(CACHE_DIR, exist_ok=True)
BASE_URL = "https://api.semanticscholar.org/graph/v1"

def s2_get(endpoint, params=None, retries=5):
    """Rate-limited GET with retry logic. Always returns a dict (never None)."""
    for attempt in range(retries):
        try:
            r = requests.get(BASE_URL + endpoint, params=params, timeout=30)
            if r.status_code == 429:
                wait = 5 * (attempt + 1)  # back off progressively
                print(f"Rate limited - waiting {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                print(f"  HTTP {r.status_code} on attempt {attempt+1}: {r.text[:200]}")
                time.sleep(3)
                continue
            time.sleep(0.2)  # be polite to the API
            return r.json()
        except requests.exceptions.Timeout:
            print(f"  Timeout on attempt {attempt+1}, retrying...")
            time.sleep(3)
        except Exception as e:
            print(f"  Error on attempt {attempt+1}: {e}")
            if attempt == retries - 1:
                raise
            time.sleep(3)
    print(f"  All retries failed for {endpoint}, returning empty result.")
    return {"data": []}

def load_or_fetch(cache_name, fetch_fn):
    """Load from cache if available, otherwise fetch and save."""
    path = os.path.join(CACHE_DIR, cache_name.replace(" ", "_") + ".json")
    if os.path.exists(path):
        print(f"  Loading from cache: {path}")
        with open(path) as f:
            return json.load(f)
    data = fetch_fn()
    with open(path, "w") as f:
        json.dump(data, f)
    return data

### Find the author

In [ ]:
def fetch_author_search():
    print(f"Searching for: {AUTHOR_NAME}")
    result = s2_get("/author/search", params={
        "query": AUTHOR_NAME,
        "fields": "authorId,name,paperCount,citationCount,hIndex",
        "limit": 5
    })
    return result["data"]

authors = load_or_fetch(f"author_search_{AUTHOR_NAME}", fetch_author_search)

print("\nTop matches:")
for i, a in enumerate(authors):
    print(f"  [{i}] {a['name']} - {a.get('paperCount', '?')} papers, "
          f"h-index {a.get('hIndex', '?')}, id={a['authorId']}")

In [ ]:
# Pick the right match from the list above (change AUTHOR_IDX if needed)
AUTHOR_IDX = 0
author = authors[AUTHOR_IDX]
AUTHOR_ID = author["authorId"]
print(f"Selected: {author['name']} (id={AUTHOR_ID})")

### Fetch papers and citation edges

We fetch ALL of the author's papers, then keep only the top `TOP_N` by citation count as our seed set. This way we focus on the most influential work, and students will recognise the famous papers. For each seed paper, we also fetch its references and citations to build the graph neighbourhood. The first run will take a while - go get a coffee!

In [ ]:
def fetch_all_papers():
    print(f"Fetching papers for {author['name']}...")
    result = s2_get(f"/author/{AUTHOR_ID}/papers", params={
        "fields": "paperId,title,year,citationCount,authors,abstract",
        "limit": 200
    })
    papers = result.get("data", [])
    print(f"  Got {len(papers)} papers.")
    return papers

all_fetched = load_or_fetch(f"all_papers_{AUTHOR_ID}", fetch_all_papers)

# Keep top N by citation count as our seed set
all_fetched_sorted = sorted(all_fetched, key=lambda p: p.get("citationCount") or 0, reverse=True)
papers = all_fetched_sorted[:TOP_N]

print(f"\nTotal papers fetched: {len(all_fetched)}")
print(f"Using top {TOP_N} by citation count. Least-cited in seed set: "
      f"{papers[-1].get('citationCount', 0)} citations")

In [ ]:
seed_ids = {p["paperId"] for p in papers}

def fetch_citation_edges():
    edges = []          # (src_id, dst_id) where src cites dst
    extra_papers = {}   # papers outside the seed set that appear as neighbours

    for i, paper in enumerate(papers):
        pid = paper["paperId"]
        print(f"  [{i+1}/{len(papers)}] {paper['title'][:60]}")

        try:
            refs = s2_get(f"/paper/{pid}/references", params={
                "fields": "paperId,title,citationCount,year", "limit": 50})
            for ref in refs.get("data", []):
                cited = ref.get("citedPaper", {})
                cited_id = cited.get("paperId")
                if cited_id:
                    edges.append((pid, cited_id))
                    if cited_id not in seed_ids:
                        extra_papers[cited_id] = cited
        except Exception as e:
            print(f"    References failed: {e}")

        try:
            cits = s2_get(f"/paper/{pid}/citations", params={
                "fields": "paperId,title,citationCount,year", "limit": 50})
            for cit in cits.get("data", []):
                citing = cit.get("citingPaper", {})
                citing_id = citing.get("paperId")
                if citing_id:
                    edges.append((citing_id, pid))
                    if citing_id not in seed_ids:
                        extra_papers[citing_id] = citing
        except Exception as e:
            print(f"    Citations failed: {e}")

    return {"edges": edges, "extra_papers": extra_papers}

citation_data = load_or_fetch(f"citations_top{TOP_N}_{AUTHOR_ID}", fetch_citation_edges)
print(f"\nCitation edges: {len(citation_data['edges'])}")
print(f"Neighbour papers: {len(citation_data['extra_papers'])}")

## Build the graph

Now let's assemble everything into a PyG `Data` object. We have two types of edges: citation edges (directed; A cites B) and co-authorship edges (undirected; two seed papers share at least one author). We also optionally enrich the node features using TF-IDF vectors computed from paper abstracts below.

In [ ]:
# Build a single node index across all papers
all_papers = {}
for p in papers:
    all_papers[p["paperId"]] = {**p, "is_seed": True}
for pid, p in citation_data["extra_papers"].items():
    if pid not in all_papers:
        all_papers[pid] = {**p, "is_seed": False}

node_ids = list(all_papers.keys())
node_idx  = {pid: i for i, pid in enumerate(node_ids)}
N = len(node_ids)

print(f"Total nodes (papers): {N}")
print(f"  Seed (top {TOP_N} by citations): {sum(1 for p in all_papers.values() if p.get('is_seed'))}")
print(f"  Neighbours (cited/citing):       {sum(1 for p in all_papers.values() if not p.get('is_seed'))}")

In [ ]:
# Node features: age, is_seed, log in-degree, log out-degree
# We deliberately do NOT include citation count - the GNN must use
# graph structure to do well, not just copy the label through.
CURRENT_YEAR = 2026

# Compute degrees from raw edge data (citation_data is already loaded)
from collections import Counter
raw_in  = Counter(dst for src, dst in citation_data["edges"] if dst in node_idx)
raw_out = Counter(src for src, dst in citation_data["edges"] if src in node_idx)

def paper_features(p, pid):
    year = p.get("year") or CURRENT_YEAR
    age  = max(float(CURRENT_YEAR - int(year)), 1.0)
    return [
        age,
        float(p.get("is_seed", False)),
        float(np.log1p(raw_in.get(pid, 0))),    # log in-degree
        float(np.log1p(raw_out.get(pid, 0))),   # log out-degree
    ]

x_base = torch.tensor(
    [paper_features(all_papers[pid], pid) for pid in node_ids],
    dtype=torch.float
)

# Labels: log citation count
y = torch.tensor(
    [np.log1p(all_papers[pid].get("citationCount") or 0) for pid in node_ids],
    dtype=torch.float
).unsqueeze(1)

print(f"Node feature matrix: {x_base.shape}  (age, is_seed, log_in_degree, log_out_degree)")
print(f"Label vector:        {y.shape}  (log citation count)")

In [ ]:
# Optional: enrich node features with TF-IDF abstract embeddings.
# TF-IDF (Term Frequency - Inverse Document Frequency) turns each abstract into
# a vector where each dimension represents a word, weighted by how distinctive
# that word is for this paper relative to all others.
# We then compress to 16 dimensions with SVD (like PCA for sparse matrices).
#
# Note: once we cover Transformers later in the course, you could replace this
# with a sentence-transformer embedding (e.g. 'allenai/scibert') for a much
# richer representation of each paper's content.

USE_ABSTRACT_FEATURES = True   # Set to False to skip
ABSTRACT_DIM = 16              # Number of SVD components to keep

if USE_ABSTRACT_FEATURES:
    abstracts = [all_papers[pid].get("abstract") or "" for pid in node_ids]
    n_with_abstract = sum(1 for a in abstracts if a)
    print(f"Papers with abstracts: {n_with_abstract} / {N}")

    tfidf = TfidfVectorizer(max_features=500, stop_words="english")
    tfidf_matrix = tfidf.fit_transform(abstracts)  # sparse (N, 500)

    svd = TruncatedSVD(n_components=ABSTRACT_DIM, random_state=42)
    abstract_features = svd.fit_transform(tfidf_matrix)  # dense (N, ABSTRACT_DIM)

    x = torch.cat([
        x_base,
        torch.tensor(abstract_features, dtype=torch.float)
    ], dim=1)
    print(f"Node feature matrix with abstracts: {x.shape}")
else:
    x = x_base
    print(f"Node feature matrix (no abstracts): {x.shape}")

In [ ]:
# Citation edges
cite_src, cite_dst = [], []
for src_id, dst_id in citation_data["edges"]:
    if src_id in node_idx and dst_id in node_idx:
        cite_src.append(node_idx[src_id])
        cite_dst.append(node_idx[dst_id])

# Co-authorship edges (between seed papers sharing at least one author)
author_to_papers = defaultdict(list)
for p in papers:
    for a in (p.get("authors") or []):
        aid = a.get("authorId")
        if aid:
            author_to_papers[aid].append(p["paperId"])

coauth_src, coauth_dst = [], []
for aid, pids in author_to_papers.items():
    for i in range(len(pids)):
        for j in range(i+1, len(pids)):
            pi, pj = pids[i], pids[j]
            if pi in node_idx and pj in node_idx:
                # Undirected - add both directions
                coauth_src += [node_idx[pi], node_idx[pj]]
                coauth_dst += [node_idx[pj], node_idx[pi]]

edge_index = torch.tensor([cite_src + coauth_src, cite_dst + coauth_dst], dtype=torch.long)
edge_type  = torch.tensor([0]*len(cite_src) + [1]*len(coauth_src), dtype=torch.long)

print(f"Citation edges:    {len(cite_src)}")
print(f"Co-author edges:   {len(coauth_src)}")
print(f"Total edges:       {edge_index.shape[1]}")

In [ ]:
# Train on seed papers that have at least 1 citation
is_seed = torch.tensor([all_papers[pid].get("is_seed", False) for pid in node_ids])
train_mask = is_seed & (y.squeeze() > 0)

graph = Data(
    x=x,
    edge_index=edge_index,
    edge_type=edge_type,
    y=y,
    train_mask=train_mask,
)

print(graph)
print(f"Training nodes: {train_mask.sum().item()}")

## Visualize the data

In [ ]:
import networkx as nx

# Build a networkx graph from seed papers only (easier to visualize)
seed_indices = [i for i, pid in enumerate(node_ids) if all_papers[pid].get("is_seed")]
seed_set = set(seed_indices)

G_vis = nx.DiGraph()
for idx in seed_indices:
    G_vis.add_node(idx)
for s, d in zip(cite_src, cite_dst):
    if s in seed_set and d in seed_set:
        G_vis.add_edge(s, d)

cit_counts = np.array([all_papers[node_ids[i]].get("citationCount") or 0 for i in seed_indices])
node_sizes = 50 + 500 * (cit_counts / (cit_counts.max() + 1))

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G_vis, seed=42, k=0.5)
nx.draw_networkx_nodes(G_vis, pos, nodelist=seed_indices,
                       node_size=node_sizes,
                       node_color=np.log1p(cit_counts), cmap="viridis", alpha=0.85)
nx.draw_networkx_edges(G_vis, pos, alpha=0.3, arrows=True, arrowsize=8)

# Label the top 10 most-cited papers
top10 = np.argsort(cit_counts)[-10:]
labels = {seed_indices[i]: all_papers[node_ids[seed_indices[i]]].get("title", "")[:30] for i in top10}
nx.draw_networkx_labels(G_vis, pos, labels=labels, font_size=6)

plt.title(f"Citation graph: {author['name']} (top {TOP_N} papers, sized by citation count)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
seed_cit  = [all_papers[pid].get("citationCount") or 0 for pid in node_ids if all_papers[pid].get("is_seed")]
seed_year = [all_papers[pid].get("year") or CURRENT_YEAR for pid in node_ids if all_papers[pid].get("is_seed")]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(seed_cit, bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Citation count")
axes[0].set_ylabel("Number of papers")
axes[0].set_title("Citation count (linear scale)")

axes[1].hist(np.log1p(seed_cit), bins=20, color="darkorange", edgecolor="white")
axes[1].set_xlabel("log(citation count + 1)")
axes[1].set_ylabel("Number of papers")
axes[1].set_title("Citation count (log scale)")

axes[2].scatter(seed_year, np.log1p(seed_cit), alpha=0.7, color="seagreen")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("log(citation count + 1)")
axes[2].set_title("Citations vs year")

plt.tight_layout()
plt.show()

We see that citation counts are VERY skewed (a power law distribution, like many things in nature!), and that older papers tend to have more citations. The year dependence is expected - a paper published last year simply hasn't had time to accumulate citations. We will revisit this when we compare the GNN rankings.

## Baseline: rank by citation count

Before we train any GNN, let's look at what a simple citation count ranking gives us. This is our baseline - the "naive" thing a librarian might do without knowing anything about graph structure.

In [ ]:
seed_papers_list = [
    {
        "title":     all_papers[pid].get("title", "Unknown"),
        "year":      all_papers[pid].get("year", 0),
        "citations": all_papers[pid].get("citationCount") or 0,
        "node_idx":  node_idx[pid],
    }
    for pid in node_ids if all_papers[pid].get("is_seed")
]

df = pd.DataFrame(seed_papers_list).sort_values("citations", ascending=False).reset_index(drop=True)
df["citation_rank"] = df.index + 1
df["age"] = df["year"].apply(lambda y: max(CURRENT_YEAR - (y or CURRENT_YEAR), 1))

print(f"Top 15 papers by citation count for {author['name']}:")
print(df[["citation_rank", "title", "year", "citations"]].head(15).to_string(index=False))

## Setting up our model infrastructure

In [ ]:
def make_mlp(
    input_size,
    sizes,
    hidden_activation="ReLU",
    output_activation=None,
    layer_norm=False,
):
    """Construct an MLP with specified fully-connected layers."""
    hidden_activation = getattr(nn, hidden_activation)()
    output_activation = getattr(nn, output_activation)() if output_activation else None
    layers = []
    sizes = [input_size] + sizes

    for i in range(len(sizes) - 2):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if layer_norm:
            layers.append(nn.LayerNorm(sizes[i + 1], elementwise_affine=False))
        layers.append(hidden_activation)

    layers.append(nn.Linear(sizes[-2], sizes[-1]))
    if output_activation:
        layers.append(output_activation)

    return nn.Sequential(*layers)


class BaseLitModel(lit.LightningModule):
    def __init__(self, lr=1e-3, weight_decay=0, lr_decay=0.95):
        super().__init__()
        self.lr = lr
        self.weight_decay = weight_decay
        self.lr_decay = lr_decay

    def training_step(self, batch, batch_idx):
        loss = self.compute_loss(batch)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('learning_rate', self.trainer.optimizers[0].param_groups[0]['lr'],
                 on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        lr_scheduler = {
            'scheduler': torch.optim.lr_scheduler.LambdaLR(
                optimizer, lr_lambda=lambda epoch: self.lr_decay ** epoch),
            'name': 'lambda_decay'
        }
        return [optimizer], [lr_scheduler]


def find_latest_directory(base_path):
    list_of_files = glob.glob(f'{base_path}/version_*')
    return max(list_of_files, key=os.path.getmtime)

def plot_metrics(directory, model_name):
    metrics_df = pd.read_csv(os.path.join(directory, 'metrics.csv'))
    train_loss_df = metrics_df[metrics_df['train_loss'].notna()]
    lr_df = metrics_df[metrics_df['learning_rate'].notna()]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_loss_df['step'], train_loss_df['train_loss'], label='Training Loss')
    plt.title(f'{model_name}: Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(lr_df['epoch'], lr_df['learning_rate'], label='Learning Rate')
    plt.title(f'{model_name}: Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.legend()

    plt.tight_layout()
    plt.show()

## Task 1: Paper ranking with a GCN

We train a Graph Convolutional Network (GCN) on the **node regression** task: predict each paper's age-corrected citation rate (citations per year) from its graph neighbourhood. The GCN only sees each paper's age and whether it is a seed paper — it must use the graph structure to do well. A paper cited by high-impact neighbours will "receive" their signal and score higher.

We will compare this against two baselines: raw citation count (ignores age) and citations-per-year (corrects for age but ignores graph structure). The GNN should in principle outperform both.

Note that for node-level tasks the whole graph is a single "batch" - there's no splitting into train/test graphs. Instead, we use a `train_mask` to specify which nodes to compute the loss on.

In [ ]:
torch.set_float32_matmul_precision('medium')
from torch.utils.data import DataLoader as TorchDataLoader

# The whole graph is one batch - wrap in a list for Lightning's DataLoader
train_loader = TorchDataLoader([graph], batch_size=1, collate_fn=lambda x: x[0])


class CitationGCN(BaseLitModel):
    def __init__(self, input_size, hidden_size, output_size, num_layers=3):
        super(CitationGCN, self).__init__(lr=1e-3, weight_decay=1e-5, lr_decay=0.99)
        self.criterion = nn.MSELoss()
        self.input_network = make_mlp(input_size, [hidden_size]*2, layer_norm=True, hidden_activation="ReLU")
        self.convs = nn.ModuleList([
            GCNConv(hidden_size, hidden_size)
            for _ in range(num_layers)
        ])
        self.output_network = make_mlp(hidden_size, [hidden_size, output_size],
                                       layer_norm=True, hidden_activation="ReLU")

    def forward(self, graph):
        x = self.input_network(graph.x)
        for conv in self.convs:
            x = x + F.relu(conv(x, graph.edge_index))  # skip connection
        return self.output_network(x)

    def compute_loss(self, batch):
        pred = self(batch)
        return self.criterion(pred[batch.train_mask], batch.y[batch.train_mask])


input_size  = graph.x.shape[1]
hidden_size = 64
output_size = 1
model = CitationGCN(input_size, hidden_size, output_size, num_layers=3)

csv_logger = CSVLogger('lightning_logs', name='citation_gcn_model')
trainer = Trainer(max_epochs=500, accelerator="gpu", logger=csv_logger)
trainer.fit(model, train_loader)

In [ ]:
base_path = 'lightning_logs/citation_gcn_model'
latest_dir = find_latest_directory(base_path)
plot_metrics(latest_dir, 'Citation GCN')

## Compare GNN ranking vs citation count

Okay, so we trained a model.  But does it actually rank papers differently from citation count?  And does it correct for year bias?  Let's find out!

In [ ]:
model.eval()
device = next(model.parameters()).device
with torch.no_grad():
    gnn_scores = model(graph.to(device)).cpu().squeeze().numpy()

df["gnn_score"] = [gnn_scores[row["node_idx"]] for _, row in df.iterrows()]
df["gnn_rank"]  = df["gnn_score"].rank(ascending=False).astype(int)
df["rank_diff"] = df["citation_rank"] - df["gnn_rank"]   # positive = GNN ranks it higher

print("Top 15 by GNN score:")
print(df.sort_values("gnn_rank")[
    ["gnn_rank", "citation_rank", "rank_diff", "title", "year", "citations"]
].head(15).to_string(index=False))

In [ ]:
print("Papers the GNN ranks significantly HIGHER (graph structure boosts them):")
print(df[df["rank_diff"] > 5].sort_values("rank_diff", ascending=False)[
    ["gnn_rank", "citation_rank", "rank_diff", "title", "year", "citations"]
].head(10).to_string(index=False))

print()
print("Papers the GNN ranks significantly LOWER (over-cited relative to network position):")
print(df[df["rank_diff"] < -5].sort_values("rank_diff")[
    ["gnn_rank", "citation_rank", "rank_diff", "title", "year", "citations"]
].head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df["year"], df["citations"], alpha=0.7, color="steelblue")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Citation count")
axes[0].set_title("Baseline: citation count vs year")

axes[1].scatter(df["year"], df["gnn_score"], alpha=0.7, color="darkorange")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("GNN score")
axes[1].set_title("GNN score vs year")

plt.suptitle(f"{author['name']}: does each ranking depend on year?", y=1.02)
plt.tight_layout()
plt.show()

# Compare how strongly each ranking depends on year
rho_cit_year, _ = spearmanr(df["year"], df["citations"])
rho_gnn_year, _ = spearmanr(df["year"], df["gnn_score"])
rho_gnn_cit,  _ = spearmanr(df["citation_rank"], df["gnn_rank"])

print(f"Spearman rho (citation count vs year): {rho_cit_year:.3f}")
print(f"Spearman rho (GNN score vs year):      {rho_gnn_year:.3f}")
print(f"Spearman rho (GNN rank vs citation rank): {rho_gnn_cit:.3f}")
print()
print("If the GNN score is less correlated with year than the raw citation count,")
print("it has learned to partially correct for the age bias in citation counts.")

## Task 2: Link prediction (future citations)

We are going to train a GNN to predict which citation edges will appear *in the future*, having only seen papers published up to `SPLIT_YEAR`.  The same idea is used to suggest new connections in social networks.

The approach:
1. Split edges into **past** (both papers published ≤ `SPLIT_YEAR`) and **future** (at least one paper published > `SPLIT_YEAR`)
2. Train a GCN on the past subgraph with a **link prediction** objective: for each pair of nodes, predict whether an edge exists
3. For negative examples, we sample random non-edges (pairs of papers that don't actually cite each other)
4. At test time, score all candidate edges involving future papers and see if the real future citations rank highly
5. Which papers does the GNN think are most likely to get new citations?

In [ ]:
# Split edges into past and future based on SPLIT_YEAR
def paper_year(pid):
    return all_papers[pid].get("year") or CURRENT_YEAR

past_src, past_dst = [], []
future_src, future_dst = [], []

for s, d in zip(cite_src, cite_dst):
    src_pid = node_ids[s]
    dst_pid = node_ids[d]
    if paper_year(src_pid) <= SPLIT_YEAR and paper_year(dst_pid) <= SPLIT_YEAR:
        past_src.append(s)
        past_dst.append(d)
    else:
        future_src.append(s)
        future_dst.append(d)

past_edge_index   = torch.tensor([past_src, past_dst], dtype=torch.long)
future_edge_index = torch.tensor([future_src, future_dst], dtype=torch.long)

print(f"Past edges   (both papers <= {SPLIT_YEAR}): {len(past_src)}")
print(f"Future edges (at least one > {SPLIT_YEAR}): {len(future_src)}")

In [ ]:
# Build the past-only graph for training
# We include co-authorship edges in full (they don't have a year)
past_coauth = torch.tensor([coauth_src, coauth_dst], dtype=torch.long)
past_all_edges = torch.cat([past_edge_index, past_coauth], dim=1) if len(coauth_src) > 0 else past_edge_index

graph_past = Data(
    x=x,
    edge_index=past_all_edges,
    num_nodes=N,
)
print(graph_past)

In [ ]:
torch.set_float32_matmul_precision('medium')

class LinkPredGCN(BaseLitModel):
    """
    GCN for link prediction. The model learns node embeddings, then
    scores each edge as the dot product of its two endpoint embeddings.
    """
    def __init__(self, input_size, hidden_size, num_layers=3):
        super(LinkPredGCN, self).__init__(lr=1e-3, weight_decay=1e-5, lr_decay=0.99)
        self.criterion = nn.BCEWithLogitsLoss()
        self.input_network = make_mlp(input_size, [hidden_size]*2, layer_norm=True, hidden_activation="ReLU")
        self.convs = nn.ModuleList([
            GCNConv(hidden_size, hidden_size)
            for _ in range(num_layers)
        ])

    def encode(self, graph):
        """Produce a hidden embedding for every node."""
        x = self.input_network(graph.x)
        for conv in self.convs:
            x = x + F.relu(conv(x, graph.edge_index))  # skip connection
        return x

    def decode(self, z, edge_index):
        """Score edges as dot product of endpoint embeddings."""
        return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=1)

    def compute_loss(self, batch):
        z = self.encode(batch)
        pos_edge = batch.edge_index
        pos_score = self.decode(z, pos_edge)
        neg_edge = negative_sampling(
            edge_index=pos_edge,
            num_nodes=batch.num_nodes,
            num_neg_samples=pos_edge.shape[1]
        )
        neg_score = self.decode(z, neg_edge)
        scores = torch.cat([pos_score, neg_score])
        labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).to(scores.device)
        return self.criterion(scores, labels)


link_model = LinkPredGCN(input_size=x.shape[1], hidden_size=64, num_layers=3)

link_loader = TorchDataLoader([graph_past], batch_size=1, collate_fn=lambda x: x[0])
csv_logger_link = CSVLogger('lightning_logs', name='link_pred_model')
trainer_link = Trainer(max_epochs=500, accelerator="gpu", logger=csv_logger_link)
trainer_link.fit(link_model, link_loader)

In [ ]:
base_path_link = 'lightning_logs/link_pred_model'
latest_dir_link = find_latest_directory(base_path_link)
plot_metrics(latest_dir_link, 'Link Prediction GCN')

In [ ]:
# Evaluate: how well does the model rank future edges over random non-edges?
link_model.eval()
device = next(link_model.parameters()).device

with torch.no_grad():
    z = link_model.encode(graph_past.to(device))

    # Score the real future edges
    if future_edge_index.shape[1] > 0:
        pos_scores = link_model.decode(z, future_edge_index.to(device)).cpu()
    else:
        print("No future edges found! Try adjusting SPLIT_YEAR to an earlier date.")
        pos_scores = torch.tensor([])

    # Score random non-edges as negatives
    neg_edge = negative_sampling(
        edge_index=future_edge_index.to(device),
        num_nodes=N,
        num_neg_samples=future_edge_index.shape[1]
    )
    neg_scores = link_model.decode(z, neg_edge).cpu()

all_scores = torch.cat([pos_scores, neg_scores]).numpy()
all_labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
auc = roc_auc_score(all_labels, all_scores)

print(f"Link prediction AUC on future edges: {auc:.3f}")
print(f"(0.5 = random chance, 1.0 = perfect)")
print(f"\nReal future edges scored: {len(pos_scores)}")
print(f"Random non-edges scored:  {len(neg_scores)}")

In [ ]:
with torch.no_grad():
    z = link_model.encode(graph_past.to(device)).cpu()

non_seed_indices = [i for i, pid in enumerate(node_ids) if not all_papers[pid].get("is_seed")]
incoming_scores = defaultdict(float)

for dst in seed_indices:
    # Batch over potential sources for efficiency
    src_batch = torch.tensor(non_seed_indices)
    dst_batch = torch.full((len(non_seed_indices),), dst)
    edge_batch = torch.stack([src_batch, dst_batch])
    scores = link_model.decode(z, edge_batch).detach().numpy()
    incoming_scores[dst] = float(scores.mean())

df["link_pred_score"] = [incoming_scores.get(row["node_idx"], 0.0) for _, row in df.iterrows()]
df["link_pred_rank"]  = df["link_pred_score"].rank(ascending=False).astype(int)

print(f"Papers the GNN thinks will attract the most future citations (after {SPLIT_YEAR}):")
print(df.sort_values("link_pred_rank")[
    ["link_pred_rank", "citation_rank", "title", "year", "citations"]
].head(15).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
diag = [1, len(df)]

axes[0].scatter(df["citation_rank"], df["gnn_rank"], alpha=0.7, c=df["year"], cmap="plasma", s=60)
axes[0].plot(diag, diag, 'k--', alpha=0.3)
axes[0].set_xlabel("Citation rank")
axes[0].set_ylabel("GNN rank")
rho_0, _ = spearmanr(df["citation_rank"], df["gnn_rank"])
axes[0].set_title(f"GNN vs citation rank (rho={rho_0:.2f})")

axes[1].scatter(df["citation_rank"], df["link_pred_rank"], alpha=0.7, c=df["year"], cmap="plasma", s=60)
axes[1].plot(diag, diag, 'k--', alpha=0.3)
axes[1].set_xlabel("Citation rank")
axes[1].set_ylabel("Link prediction rank")
rho_1, _ = spearmanr(df["citation_rank"], df["link_pred_rank"])
axes[1].set_title(f"Link pred vs citation rank (rho={rho_1:.2f})")

plt.suptitle(f"{author['name']}: comparing ranking methods (colored by year)", y=1.02)
plt.tight_layout()
plt.show()

rho_gnn_link, _ = spearmanr(df["gnn_rank"], df["link_pred_rank"])
print(f"GNN rank vs link pred rank: rho = {rho_gnn_link:.3f}")

The two tasks are asking very different questions of the graph:
  Task 1: given the full citation network, which papers are most central?
  Task 2: given only past edges, which future edges are most likely?

Questions to think about:
  - Why might the two GNN rankings disagree with each other?
  - What information would help the link prediction model most?
  - Is predicting citations fundamentally harder than classifying molecules? Why?

## Try your own experiments!

Now it's your turn. Some things to try:
- **Different authors**: change `AUTHOR_NAME` at the top and re-run. How does the ranking structure differ between a young researcher and an established one? Does the link prediction AUC change?
- **Different split years**: try `SPLIT_YEAR = 2015` or `SPLIT_YEAR = 2022`. How does the AUC change? What does this tell you about how far into the future a GNN can predict?
- **Remove co-authorship edges**: comment out the co-authorship edge block and rebuild the graph. Does the collaboration structure add useful information for either task?
- **More or fewer GCN layers**: try `num_layers=1` or `num_layers=5`. More layers = each node sees a wider neighbourhood. Is there an optimal depth for this graph?
- **Abstract embeddings**: we used TF-IDF to encode paper content. If you want to experiment with the transformer idea we described at the end, try replacing the TF-IDF features with a sentence-transformer embedding (e.g. `allenai/scibert`) for a much richer representation of each paper's physics content.